In [1]:
import sqlite3
import random
import time
import requests
import pandas as pd
from datetime import date, timedelta
from bs4 import BeautifulSoup

In [2]:
random.seed(7)
conn = sqlite3.connect("retail.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS orders")
cur.execute("DROP TABLE IF EXISTS customers")

cur.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    segment TEXT
)
""")

cur.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    amount REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cities = ["Seoul", "Busan", "Incheon", "Daegu", "Gwangju"]
segments = ["Retail", "Wholesale", "Online"]
names = [
    "Kim Minji", "Park Jisoo", "Lee Jaehyun", "Choi Seoyeon",
    "Jung Hyunwoo", "Kang Yuna", "Yoon Doyoon", "Han Soomin",
    "Oh Jiwoo", "Shin Taeyang", "Baek Narae", "Song Junho",
    "Moon Hana", "Ahn Jiho", "Jang Yerin"
]

customers = [
    (i, name, random.choice(cities), random.choice(segments))
    for i, name in enumerate(names, start=1)
]
cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)

start_date = date(2026, 1, 1)
orders = []
order_id = 1

for customer in customers:
    for _ in range(random.randint(2, 8)):
        d = start_date + timedelta(days=random.randint(0, 270))
        amount = round(random.uniform(15000, 450000), -2)
        orders.append((order_id, customer[0], d.isoformat(), amount))
        order_id += 1

cur.executemany("INSERT INTO orders VALUES (?,?,?,?)", orders)
conn.commit()

print("Database created successfully!")

Database created successfully!


In [3]:
def customers_in_city_segment(conn, city, segment):
    query = """
        SELECT name, city, segment
        FROM customers
        WHERE city = ? AND segment = ?
    """
    return pd.read_sql(query, conn, params=(city, segment))

print("\nProblem 1")
print(customers_in_city_segment(conn, "Seoul", "Online"))


Problem 1
         name   city segment
0   Kang Yuna  Seoul  Online
1  Song Junho  Seoul  Online


In [4]:
def orders_with_customer_names(conn):
    query = """
        SELECT c.name, o.order_date, o.amount
        FROM customers c
        INNER JOIN orders o ON c.customer_id = o.customer_id
        LIMIT 10
    """
    return pd.read_sql(query, conn)

print("\nProblem 2")
print(orders_with_customer_names(conn))


Problem 2
         name  order_date    amount
0   Kim Minji  2026-01-26  439700.0
1   Kim Minji  2026-01-24  257100.0
2   Kim Minji  2026-03-10  141000.0
3   Kim Minji  2026-03-15  250200.0
4   Kim Minji  2026-06-07  258700.0
5  Park Jisoo  2026-04-03   59800.0
6  Park Jisoo  2026-04-07  177000.0
7  Park Jisoo  2026-02-02  260500.0
8  Park Jisoo  2026-04-16  230900.0
9  Park Jisoo  2026-08-07  353100.0


In [5]:
def high_spending_customers(conn, min_total=300000):
    query = """
        SELECT c.name, SUM(o.amount) AS total_spent
        FROM customers c
        INNER JOIN orders o ON c.customer_id = o.customer_id
        GROUP BY c.customer_id, c.name
        HAVING SUM(o.amount) > ?
        ORDER BY total_spent DESC
    """
    return pd.read_sql(query, conn, params=(min_total,))

print("\nProblem 3")
print(high_spending_customers(conn))


Problem 3
            name  total_spent
0     Jang Yerin    2167100.0
1    Yoon Doyoon    1609600.0
2     Baek Narae    1559500.0
3     Park Jisoo    1523300.0
4       Oh Jiwoo    1437300.0
5     Song Junho    1359000.0
6      Kim Minji    1346700.0
7       Ahn Jiho    1129700.0
8      Moon Hana     895700.0
9   Jung Hyunwoo     861200.0
10  Shin Taeyang     619200.0
11   Lee Jaehyun     613200.0
12    Han Soomin     427900.0
13  Choi Seoyeon     374600.0


In [6]:
def top_n_customers(conn, n=3):
    query = """
        SELECT c.name, SUM(o.amount) AS total_spent
        FROM customers c
        INNER JOIN orders o ON c.customer_id = o.customer_id
        GROUP BY c.customer_id, c.name
        ORDER BY total_spent DESC
        LIMIT ?
    """
    return pd.read_sql(query, conn, params=(n,))

print("\nProblem 4")
print(top_n_customers(conn, 3))


Problem 4
          name  total_spent
0   Jang Yerin    2167100.0
1  Yoon Doyoon    1609600.0
2   Baek Narae    1559500.0


In [7]:
def find_orders_by_customer(conn, customer_name):
    query = """
        SELECT o.order_id, o.order_date, o.amount
        FROM orders o
        INNER JOIN customers c ON o.customer_id = c.customer_id
        WHERE c.name = ?
    """
    cur = conn.cursor()
    cur.execute(query, (customer_name,))
    return cur.fetchall()

print("\nProblem 5")
print(find_orders_by_customer(conn, "Kim Minji"))


Problem 5
[(1, '2026-01-26', 439700.0), (2, '2026-01-24', 257100.0), (3, '2026-03-10', 141000.0), (4, '2026-03-15', 250200.0), (5, '2026-06-07', 258700.0)]


In [8]:
def monthly_revenue(conn):
    query = """
        SELECT strftime('%Y-%m', order_date) AS month,
               SUM(amount) AS revenue
        FROM orders
        GROUP BY strftime('%Y-%m', order_date)
        ORDER BY month
    """
    return pd.read_sql(query, conn)

print("\nProblem 6")
print(monthly_revenue(conn))



Problem 6
     month    revenue
0  2026-01  1605200.0
1  2026-02  1924900.0
2  2026-03  1836600.0
3  2026-04  2277600.0
4  2026-05  1005400.0
5  2026-06  2817100.0
6  2026-07  1715100.0
7  2026-08  1906900.0
8  2026-09  1097800.0


In [9]:
def get_current_weather(latitude, longitude):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": "true"
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        return response.json().get("current_weather")
    except (requests.RequestException, ValueError) as error:
        print("Weather error:", error)
        return None

print("\nProblem 7")
print(get_current_weather(37.5665, 126.9780))



Problem 7
{'time': '2026-10-12T02:45', 'interval': 900, 'temperature': 23.1, 'windspeed': 3.2, 'winddirection': 117, 'is_day': 1, 'weathercode': 1}


In [10]:

def weather_for_cities(cities):
    rows = []

    for city_name, latitude, longitude in cities:
        weather = get_current_weather(latitude, longitude)

        if weather is not None:
            row = {"city": city_name}
            row.update(weather)
            rows.append(row)

    return pd.DataFrame(rows)

test_cities = [
    ("Seoul", 37.5665, 126.9780),
    ("Busan", 35.1796, 129.0756),
    ("Tokyo", 35.6762, 139.6503)
]

print("\nProblem 8")
print(weather_for_cities(test_cities))





Problem 8
    city              time  interval  temperature  windspeed  winddirection  \
0  Seoul  2026-10-12T02:45       900         23.1        3.2            117   
1  Busan  2026-10-12T02:45       900         24.0        8.4             80   
2  Tokyo  2026-10-12T02:45       900         23.2        6.5             19   

   is_day  weathercode  
0       1            1  
1       1            2  
2       1            0  


In [11]:

def weather_for_cities_paced(cities, delay_seconds=1):
    rows = []

    for city_name, latitude, longitude in cities:
        weather = get_current_weather(latitude, longitude)

        if weather is not None:
            row = {"city": city_name}
            row.update(weather)
            rows.append(row)

        time.sleep(delay_seconds)

    return pd.DataFrame(rows)

print("\nProblem 9")
print(weather_for_cities_paced(test_cities, delay_seconds=1))


Problem 9
    city              time  interval  temperature  windspeed  winddirection  \
0  Seoul  2026-10-12T02:45       900         23.1        3.2            117   
1  Busan  2026-10-12T02:45       900         24.0        8.4             80   
2  Tokyo  2026-10-12T02:45       900         23.2        6.5             19   

   is_day  weathercode  
0       1            1  
1       1            2  
2       1            0  


In [13]:
def scrape_book_titles_and_prices():
    url = "https://books.toscrape.com/"
    headers = {"User-Agent": "KDU-DataScience-Student"}

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        books = []

        for book in soup.select("article.product_pod"):
            title_element = book.select_one("h3 a")
            price_element = book.select_one("p.price_color")

            if title_element and price_element:
                books.append({
                    "title": title_element.get("title", ""),
                    "price": price_element.get_text(strip=True)
                })

        return pd.DataFrame(books)

    except requests.RequestException as error:
        print("Scraping error:", error)
        return pd.DataFrame(columns=["title", "price"])

print("\nOptional: Web Scraping")
print(scrape_book_titles_and_prices())


conn.close()



Optional: Web Scraping
                                                title    price
0                                A Light in the Attic  Â£51.77
1                                  Tipping the Velvet  Â£53.74
2                                          Soumission  Â£50.10
3                                       Sharp Objects  Â£47.82
4               Sapiens: A Brief History of Humankind  Â£54.23
5                                     The Requiem Red  Â£22.65
6   The Dirty Little Secrets of Getting Your Dream...  Â£33.34
7   The Coming Woman: A Novel Based on the Life of...  Â£17.93
8   The Boys in the Boat: Nine Americans and Their...  Â£22.60
9                                     The Black Maria  Â£52.15
10     Starving Hearts (Triangular Trade Trilogy, #1)  Â£13.99
11                              Shakespeare's Sonnets  Â£20.66
12                                        Set Me Free  Â£17.46
13  Scott Pilgrim's Precious Little Life (Scott Pi...  Â£52.29
14                          Rip